# Phase 6: Deep Learning (1D-CNN) for Arabic Document Classification
### Khaleej (SANAD) Dataset Benchmark — PyTorch Implementation for Google Colab

This self-contained notebook trains a multi-scale 1D Convolutional Neural Network (CNN) on preprocessed Arabic news articles from the Khaleej dataset.

#### Architecture Overview:
- **Embedding Layer**: Dense learned word representations ($D = 128$).
- **Parallel 1D Convolutions**: Multiple filter window sizes ($k \in \{3, 5, 7\}$) to detect sub-word affixes, tri-consonantal roots (جذور ثلاثية), and compound multi-word entities.
- **Global Max-Pooling over Time**: Extracts the most salient lexical and phrasal activations across the document.
- **Classification Head**: Dropout ($p=0.5$), Batch Normalization, and Softmax over 7 Khaleej categories.
- **Zero Data Leakage**: Vocabulary is fitted strictly on the training partition.

In [ ]:
# Cell 1: Environment Setup & GPU Verification
import csv
import time
from collections import Counter
from pathlib import Path

import joblib
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset

# Set seeds for determinism
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("=" * 60)
print(f"[1/7] Environment initialized. Device: {device}")
if torch.cuda.is_available():
    print(f"      GPU: {torch.cuda.get_device_name(0)}")
    print(f"      VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("      Running on CPU. Select: Runtime -> Change runtime type -> T4 GPU.")
print("=" * 60)

In [ ]:
# Cell 2: Data Ingestion (Auto-locates uploaded file or provides upload button)
candidate_paths = [
    Path("light_normalized_dev.joblib"),
    Path("/content/light_normalized_dev.joblib"),
    Path("data/processed/light_normalized_dev.joblib"),
    Path("/content/data/processed/light_normalized_dev.joblib"),
    Path("/content/drive/MyDrive/arabic-clf/data/processed/light_normalized_dev.joblib"),
    Path("/content/drive/MyDrive/light_normalized_dev.joblib"),
]

found_path = None
for p in candidate_paths:
    if p.is_file():
        found_path = p
        break

# If not found, trigger interactive Colab file upload
if found_path is None:
    try:
        from google.colab import files
        print("Dataset not found. Please upload 'light_normalized_dev.joblib':")
        uploaded = files.upload()
        for fname in uploaded:
            if "light_normalized_dev" in fname:
                found_path = Path(fname)
                break
    except ImportError:
        pass

if found_path is None or not found_path.is_file():
    raise FileNotFoundError(
        "CRITICAL: 'light_normalized_dev.joblib' was not found!\n"
        "To fix: Folder icon (left) -> Upload 'light_normalized_dev.joblib', "
        "then re-run this cell."
    )

print(f"[2/7] Loading preprocessed dataset from: {found_path}")
payload = joblib.load(found_path)
texts = payload["texts"]
labels = payload["labels"]
print(f"      Loaded {len(texts):,} articles across {len(set(labels))} categories.")

In [ ]:
# Cell 3: Split Protocol & Vocabulary Pipeline (Zero Leakage)
train_texts, val_texts, train_labels, val_labels = train_test_split(
    texts, labels, test_size=0.20, random_state=SEED, stratify=labels
)
print(f"[3/7] Dev split: {len(train_texts):,} train | {len(val_texts):,} val articles")

class TextVocabulary:
    def __init__(self, pad_token="<pad>", unk_token="<unk>"):
        self.pad_token = pad_token
        self.unk_token = unk_token
        self.pad_idx = 0
        self.unk_idx = 1
        self.token_to_idx = {self.pad_token: self.pad_idx, self.unk_token: self.unk_idx}

    def fit(self, texts, max_vocab_size=50000, min_freq=2):
        counter = Counter()
        for t in texts:
            counter.update(t.split())
        for token, count in counter.most_common():
            if len(self.token_to_idx) >= max_vocab_size:
                break
            if count >= min_freq and token not in self.token_to_idx:
                self.token_to_idx[token] = len(self.token_to_idx)
        return self

    def encode(self, text, max_len=512):
        tokens = text.split()[:max_len]
        ids = [self.token_to_idx.get(t, self.unk_idx) for t in tokens]
        if len(ids) < max_len:
            ids.extend([self.pad_idx] * (max_len - len(ids)))
        return ids

    def __len__(self):
        return len(self.token_to_idx)

# Fit vocabulary ONLY on train split (prevent leakage)
vocab = TextVocabulary().fit(train_texts, max_vocab_size=50000, min_freq=2)
print(f"      Vocabulary constructed: {len(vocab):,} unique words.")

class ArabicDataset(Dataset):
    def __init__(self, texts, labels, vocab, max_len=512):
        self.encoded = [vocab.encode(t, max_len=max_len) for t in texts]
        self.labels = list(labels)

    def __len__(self):
        return len(self.encoded)

    def __getitem__(self, idx):
        x_t = torch.tensor(self.encoded[idx], dtype=torch.long)
        y_t = torch.tensor(self.labels[idx], dtype=torch.long)
        return x_t, y_t

BATCH_SIZE = 64
MAX_LEN = 512

train_ds = ArabicDataset(train_texts, train_labels, vocab, max_len=MAX_LEN)
val_ds = ArabicDataset(val_texts, val_labels, vocab, max_len=MAX_LEN)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)
print(f"      DataLoaders: {len(train_loader)} train batches, {len(val_loader)} val batches.")

In [ ]:
# Cell 4: 1D-CNN Architecture Definition
class ArabicCNN_1D(nn.Module):
    def __init__(
        self,
        vocab_size,
        embedding_dim=128,
        num_filters=128,
        filter_sizes=(3, 5, 7),
        num_classes=7,
        dropout=0.5,
    ):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        self.convs = nn.ModuleList([
            nn.Conv1d(
                in_channels=embedding_dim,
                out_channels=num_filters,
                kernel_size=k,
                padding=k // 2,
            )
            for k in filter_sizes
        ])
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(num_filters * len(filter_sizes), num_classes)

    def forward(self, x):
        emb = self.embedding(x).permute(0, 2, 1)  # (batch, embedding_dim, seq_len)
        pooled = [F.adaptive_max_pool1d(F.relu(conv(emb)), 1).squeeze(2) for conv in self.convs]
        h = self.dropout(torch.cat(pooled, dim=1))
        return self.fc(h)

model = ArabicCNN_1D(
    vocab_size=len(vocab),
    embedding_dim=128,
    num_filters=128,
    filter_sizes=(3, 5, 7),
    num_classes=7,
    dropout=0.5,
).to(device)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"[4/7] ArabicCNN_1D Model Created. Trainable parameters: {total_params:,}")

In [ ]:
# Cell 5: Training & Validation Loop with Early Stopping
EPOCHS = 15
PATIENCE = 3
LEARNING_RATE = 1e-3

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

best_macro_f1 = -1.0
patience_counter = 0
best_state = None
history = {"train_loss": [], "val_loss": [], "val_f1": []}
start_train_time = time.perf_counter()

print(f"[5/7] Starting CNN Training: {EPOCHS} epochs (EarlyStopping patience={PATIENCE})...")
print("-" * 80)

for epoch in range(1, EPOCHS + 1):
    t_start_epoch = time.perf_counter()
    model.train()
    total_train_loss = 0.0
    for x_b, y_b in train_loader:
        x_b, y_b = x_b.to(device), y_b.to(device)
        optimizer.zero_grad()
        logits = model(x_b)
        loss = criterion(logits, y_b)
        loss.backward()
        optimizer.step()
        total_train_loss += loss.item() * len(y_b)
    avg_train_loss = total_train_loss / len(train_ds)

    model.eval()
    total_val_loss = 0.0
    all_preds, all_targets = [], []
    with torch.no_grad():
        for x_b, y_b in val_loader:
            x_b, y_b = x_b.to(device), y_b.to(device)
            logits = model(x_b)
            loss = criterion(logits, y_b)
            total_val_loss += loss.item() * len(y_b)
            all_preds.extend(torch.argmax(logits, dim=1).cpu().tolist())
            all_targets.extend(y_b.cpu().tolist())
    avg_val_loss = total_val_loss / len(val_ds)
    val_f1 = f1_score(all_targets, all_preds, average="macro", zero_division=0)
    val_acc = accuracy_score(all_targets, all_preds)
    epoch_time = time.perf_counter() - t_start_epoch

    history["train_loss"].append(avg_train_loss)
    history["val_loss"].append(avg_val_loss)
    history["val_f1"].append(val_f1)

    print(
        f"Epoch [{epoch:02d}/{EPOCHS:02d}] ({epoch_time:.1f}s) | "
        f"Train Loss: {avg_train_loss:.4f} | "
        f"Val Loss: {avg_val_loss:.4f} | "
        f"Val Acc: {val_acc:.4f} | "
        f"Val Macro-F1: {val_f1:.4f}",
        flush=True,
    )

    if val_f1 > best_macro_f1:
        best_macro_f1 = val_f1
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        patience_counter = 0
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(f"\n--> Early stopping at epoch {epoch} (best Val F1: {best_macro_f1:.4f})")
            break

total_train_sec = time.perf_counter() - start_train_time
print(f"\nTraining complete in {total_train_sec:.2f}s. Best Val Macro-F1: {best_macro_f1:.4f}")

In [ ]:
# Cell 6: Final Evaluation, Latency Measurement, and 14-Column CSV Logging
if best_state is None:
    best_state = model.state_dict()

model.load_state_dict(best_state)
model.eval()

all_preds, all_targets = [], []
start_infer = time.perf_counter()
with torch.no_grad():
    for x_b, y_b in val_loader:
        x_b, y_b = x_b.to(device), y_b.to(device)
        logits = model(x_b)
        all_preds.extend(torch.argmax(logits, dim=1).cpu().tolist())
        all_targets.extend(y_b.cpu().tolist())
infer_sec = time.perf_counter() - start_infer
infer_ms_per_doc = (infer_sec / len(val_ds)) * 1000.0

final_acc = accuracy_score(all_targets, all_preds)
final_prec = precision_score(all_targets, all_preds, average="macro", zero_division=0)
final_rec = recall_score(all_targets, all_preds, average="macro", zero_division=0)
final_f1 = f1_score(all_targets, all_preds, average="macro", zero_division=0)

print("=" * 60)
print("[6/7] FINAL VALIDATION BENCHMARK EVALUATION")
print("=" * 60)
print(f"Accuracy:          {final_acc:.6f}")
print(f"Macro-Precision:   {final_prec:.6f}")
print(f"Macro-Recall:      {final_rec:.6f}")
print(f"Macro-F1:          {final_f1:.6f}")
print(f"Inference Latency: {infer_ms_per_doc:.4f} ms/doc")
print("=" * 60)

COLUMNS = [
    "run_id", "model", "representation", "dim_reduction", "preprocessing",
    "noise_level", "seed", "split", "accuracy", "macro_precision",
    "macro_recall", "macro_f1", "train_time_sec", "inference_ms_per_doc"
]

csv_path = Path("results/results_dl.csv")
csv_path.parent.mkdir(parents=True, exist_ok=True)
file_exists = csv_path.is_file() and csv_path.stat().st_size > 0

row = {
    "run_id": "ArabicCNN_1D_word_embeddings_none_light_normalized_val",
    "model": "ArabicCNN_1D",
    "representation": "word_embeddings",
    "dim_reduction": "none",
    "preprocessing": "light_normalized",
    "noise_level": 0.0,
    "seed": SEED,
    "split": "dev_val_split",
    "accuracy": round(final_acc, 6),
    "macro_precision": round(final_prec, 6),
    "macro_recall": round(final_rec, 6),
    "macro_f1": round(final_f1, 6),
    "train_time_sec": round(total_train_sec, 4),
    "inference_ms_per_doc": round(infer_ms_per_doc, 4),
}

with csv_path.open("a", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=COLUMNS)
    if not file_exists:
        writer.writeheader()
    writer.writerow(row)

print(f"Results logged to {csv_path}")
checkpoint_dir = Path("models_saved")
checkpoint_dir.mkdir(parents=True, exist_ok=True)
ckpt_file = checkpoint_dir / "ArabicCNN_1D_best_weights.pt"
torch.save({"state_dict": best_state, "vocab": vocab.token_to_idx}, ckpt_file)
print(f"Best checkpoint saved to: {ckpt_file}")

In [ ]:
# Cell 7: Loss & Macro-F1 Visualization
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4), dpi=150)
epochs_range = range(1, len(history["train_loss"]) + 1)

ax1.plot(epochs_range, history["train_loss"], label="Train Loss", color="#4C72B0", lw=2)
ax1.plot(epochs_range, history["val_loss"], label="Validation Loss", color="#C44E52", lw=2)
ax1.set_xlabel("Epoch", fontweight="bold")
ax1.set_ylabel("Cross Entropy Loss", fontweight="bold")
ax1.set_title("Learning Curves", fontweight="bold")
ax1.legend()
ax1.grid(True, linestyle="--", alpha=0.5)

ax2.plot(epochs_range, history["val_f1"], label="Val Macro-F1", color="#55A868", lw=2)
ax2.set_xlabel("Epoch", fontweight="bold")
ax2.set_ylabel("Macro-F1 Score", fontweight="bold")
ax2.set_title("Validation F1 Progression", fontweight="bold")
ax2.legend()
ax2.grid(True, linestyle="--", alpha=0.5)

plt.tight_layout()
plt.show()
print("[7/7] Visualization complete.")